# AI Guardrails Tutorial
## Module 2: Introduction to Guardrails AI

This module introduces the **Guardrails AI** framework, a powerful runtime validation system designed to defend LLM applications against the vulnerability demonstrated in Module 1: Unconstrained LLMs and Prompt Injection.

## 2.1 Architectural Concepts: Validators, Guards, and Rails

Guardrails AI provides a layered defense system for LLM applications. Let's understand the core concepts.

### What is a Guardrail?

**Guardrails** are runtime validation mechanisms that check content against predefined rules. They operate at different points in the LLM pipeline:

#### The Three Layers of Defense

| Layer | Direction | Purpose | Example |
|-------|-----------|---------|----------|
| **Input Guard** | User â†’ LLM | Block malicious requests | Detect prompt injection |
| **Output Guard** | LLM â†’ User | Sanitize responses | Prevent code execution |
| **Process Rail** | Internal | Monitor and constrain behavior | Rate limiting, logging |

### The Guardrails Framework Architecture

```text
+----------------------------------------------------------------+
|                        Application Layer                        |
|  (Your business logic, agent orchestration, etc.)              |
+----------------------------------------------------------------+
                            |
                            v
+----------------------------------------------------------------+
|                      Output Guards                               |
|  (Validate and sanitize what the model generates)               |
|  - Toxicity detection                                           |
|  - JSON structure validation                                    |
|  - Fact-checking                                                |
+----------------------------------------------------------------+
                            ^
                            | LLM
+----------------------------------------------------------------+
|                      LLM Layer                                   |
|  (Your language model: OpenAI, Ollama, etc.)                    |
+----------------------------------------------------------------+
                            |
                            v
+----------------------------------------------------------------+
|                      Input Guards                                |
|  (Screen user input before it reaches the LLM)                  |
|  - Prompt injection detection                                   |
|  - PII/PI data detection                                        |
|  - Content policy checks                                        |
+----------------------------------------------------------------+


### Why Runtime Validation is Necessary

Traditional programming relies on compile-time checks and type safety. However, LLMs are fundamentally different:

1. **Non-deterministic outputs**: The same input can produce different outputs
2. **Semantic understanding**: Text meaning isn't captured by regex patterns
3. **Context sensitivity**: Guards must understand context, not just content
4. **Emergent behaviors**: Models can exhibit unexpected capabilities

## 2.2 Environment Setup & Initialization

Now let's set up Guardrails AI and prepare our environment.

In [21]:
# =============================================================================
# Initialize
# =============================================================================
!uv pip install python-dotenv --link-mode=copy

Checked 1 package in 6ms


### Dependencies are managed via `pyproject.toml`
The packages `guardrails-ai` and `guardrails-hub` are listed in the project dependencies and installed automatically with `uv sync`.

### Key Validators for Module 2
We'll focus on validators that address the prompt injection vulnerability from Module 1:

1. **prompt_injection** - Detects attempts to override system instructions
2. **system_prompt_leak** - Prevents accidental disclosure of internal prompts
3. **code_injection** - Detects embedded executable code


## 2.3 Custom Validator Examples
Let's start by defining custom validators to understand the Guardrails API.

### As a Function
A custom validator is a function that receives the input value and metadata, then returns either `PassResult`, `FailResult`, or raises an exception.

In [22]:
# =============================================================================
# Define a custom validator using register_validator
# Note: Validators must be classes inheriting from Validator, not functions
# =============================================================================
from typing import Dict
from guardrails.validator_base import Validator, register_validator, PassResult, FailResult, ValidationResult
from guardrails import Guard


In [23]:
@register_validator(name="min_length_check", data_type="string")
class MinLengthValidator(Validator):
    """
    Custom validator to ensure string has minimum length
    """
    def __init__(self, min_length: int = 5):
        super().__init__()
        self.min_length = min_length

    def _validate(self, value: str, metadata: Dict) -> ValidationResult:
        str_value = str(value) if value else ""
        if len(str_value) < self.min_length:
            return FailResult(
                error_message=f"Input string must be at least {self.min_length} characters long"
            )
        return PassResult()


In [ ]:
# Test the validator by calling it directly
print("Testing validator with different inputs:")
print("=" * 50)

# Test 1: Valid input
try:
    validator = MinLengthValidator(min_length=5)
    result = validator.validate("This is properly lengthed text", metadata={})
    if isinstance(result, PassResult):
        print("✓ Valid input passed")
    else:
        print(f"✗ Valid input incorrectly failed: {result.error_message}")
except Exception as e:
    print(f"✗ Unexpected error: {e}")

print()

# Test 2: Short input (should fail)
try:
    validator = MinLengthValidator(min_length=5)
    result = validator.validate("hi", metadata={})
    if isinstance(result, FailResult):
        print(f"✓ Short input correctly rejected: {result.error_message}")
    else:
        print(f"✗ Short input incorrectly passed")
except Exception as e:
    print(f"✗ Unexpected error: {e}")


Testing validator with different inputs:
Checkmark Valid input passed

Checkmark Short input correctly rejected: Input string must be at least 5 characters long


## 2.4 Using Guards from Guardrails AI
Now let's use the Guard class to combine multiple validators together.

In [ ]:
# =============================================================================
# Import Guardrails and validate using a Guard object
# =============================================================================
from guardrails import Guard
import guardrails as glrs


### Using `Guard.validate()`
You can create a Guard, add validators to it, and then validate input data.

In [ ]:
# Create a guard with our custom validator
guard = (
    Guard()
    .use(MinLengthValidator(min_length=5), on_fail="exception")
) 

# Test validation
test_input = "short"
try:
    result = guard.validate(test_input)
    print("Cross Validation unexpectedly passed for '%s'" % test_input)
except Exception as e:
    print("Checkmark Validation correctly rejected '%s': %s" % (test_input, e))


In [ ]:
# Test with valid input
valid_input = "This is properly lengthed text"
try:
    result = guard.validate(valid_input)
    print("Checkmark Validation passed for valid input")
except Exception as e:
    print(f"Cross Validation unexpectedly failed: {e}")


## 2.5 Guardrails Hub
The Guardrails Hub provides pre-built validators that can be installed or downloaded.

In [ ]:
# =============================================================================
# Check if Guardrails Hub client is available
# =============================================================================
try:
    from guardrails.hub import guardrail_loader
    
    print("\u2713 Guardrails Hub client loaded successfully")
    print("=" * 50)
    print("\u2713 Security-focused validators from the hub:")
    print("  - prompt_injection")
    print("  - system_prompt_leak")
    print("  - code_injection")
    print("  - toxic_language")
    print("  - detect_pii")
    print("=" * 50)
    print("\nTo use these validators:")
    print("  pip install guardrails-ai-prompt-injection")
    print("\nOr from the hub:")
    print("  guardrails download hub://guardrails/prompt_injection")
except ImportError as e:
    print(f"\u26a0️  Guardrails Hub client not available: {e}")
    print("\nThis is expected in some environments.")

## Summary: What We Achieved in Module 2

### Key Concepts Learned

1. **Guardrails AI Architecture**
   - Input guards protect against malicious user inputs
   - Output guards sanitize model responses
   - Rails control the flow between components

2. **Validator Creation**
   - Defined custom validators using `@register_validator` decorator
   - Validators work with any data type
   - Return `PassResult` or `FailResult` objects

3. **Guard Implementation**
   - Created Guards to combine multiple validators
   - Used `Guard.validate()` to validate input data
   - Configured on_fail actions for failed validations

### Next Steps in Module 3
In the next module, we'll integrate these input guards to detect and block:
- Prompt injection attacks from Module 1
- System prompt leaks
- PII/PI data leakage